## Building RAG

## Contract Intelligence Stack

In [ ]:
from wrangle_data import load_and_chunk, build_retriever, build_context
from llm_settings import extract_contract_data, ContractExtraction
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from pathlib import Path

# Initialize LLM
llm = ChatOllama(
        model = "qwen2.5:14b",
        num_gpu=-1,
        num_thread=6
    )

# Define system prompt for contract extraction
SYSTEM_PROMPT = """
    You are a contract extraction assistant. Extract information ONLY from the given contract text.

    Rules:
    - Return JSON matching the ContractExtraction schema.
    - If a value is missing, return null.
    - Do NOT infer or guess.
    - Extract exact quotes wherever possible.
    - Dates must be in ISO 8601 format (YYYY-MM-DD).
    - Currency must be ISO 4217 (USD, EUR, etc.).
    - Language must be ISO 639 (DE, EN, etc.)
    - Separate amount_value and amount_currency.
    - If multiple values exist, pick the primary contractual one.
    - For each section, clearly map text to the schema field.

    Field mapping instructions:
    - language: try to map the language by assessing the language used in the document
    - parties: list of all parties with name, role (customer, vendor, licensor, etc.), address, country
    - commercial_terms: extract amount_value, amount_currency, billing_frequency, taxes_included, due_days, late_fee, late_interest, payment_method
    - timeline: effective_date, start_date, end_date, term_months, auto_renewal, renewal_notice_days, termination_notice_days
    - legal_terms: governing_law, jurisdiction, confidentiality, data_protection, indemnification, limitation_of_liability, ip_ownership, exclusivity, assignment_restrictions, force_majeure
    - scope_summary: a short summary of the contract scope
    - services_or_deliverables: list of services or deliverables
    - obligations_customer: list of obligations of the customer
    - obligations_vendor: list of obligations of the vendor
    - sla_or_support: any SLA or support information
    - penalty_or_breach_clauses: list of clauses about penalties or breaches
    - risk_flags: list of potential risks mentioned
    - signature_date: date of signing
    - signed_by: list of names who signed
    - evidence_quotes: exact textual quotes used to extract any data
    - confidence: a float between 0 and 1 indicating extraction confidence
    """

# Define human prompt template
HUMAN_PROMPT = """
    Extract the contract data from this text:

    {context}

    Return the full schema.
    """

# Combine system and human prompts into a ChatPromptTemplate
prompt = ChatPromptTemplate.from_messages(
        [
            ("system", SYSTEM_PROMPT),
            ("human", HUMAN_PROMPT),
        ]
    )

# List all PDF files in the target directory
files = [x for x in Path("./Contracts_30_English").rglob("*.pdf")]

# Dictionary to store extracted contract data
contract_data = {}

# Process each contract file
for file in files:
    print(file) # log current file being processed

    # Step 1: Load PDF and split into chunks
    chunks = load_and_chunk(file)

    # Step 2: Build a retriever from chunks for similarity search
    retriever = build_retriever(chunks)

    # Step 3: Build a context string by querying the retriever
    context = build_context(retriever)

    # Step 4: Create extraction chain with structured output
    extract_chain = prompt | llm.with_structured_output(ContractExtraction)

    # Step 5: Extract structured contract data
    data = extract_contract_data(context, extract_chain, Path(file).stem)

    # Step 6: Store extracted data keyed by file 
    contract_data[Path(file).stem] = data

### Pandas Dataframe
#### Extract some of the information from the previous JSON output. Then create a pandas dataframe.

In [21]:
import pandas as pd

def flatten_contract(contract, contract_id=None):
    row = {}

    # Optional: ID aus äußerem Dict
    row["contract_id"] = contract_id

    # Basic
    row["file_name"] = contract.get("file_name")
    row["contract_type"] = contract.get("contract_type")
    row["language"] = contract.get("language")
    row["signature_date"] = contract.get("signature_date")
    row["confidence"] = contract.get("confidence")

    # Parties (inkl. Adresse)
    for p in contract.get("parties", []):
        role = p.get("role")
        if role == "BORROWER":
            row["borrower_name"] = p.get("name")
            row["borrower_country"] = p.get("country")
            row["borrower_address"] = p.get("address")
        elif role == "LENDER":
            row["lender_name"] = p.get("name")
            row["lender_country"] = p.get("country")
            row["lender_address"] = p.get("address")

    # Commercial Terms
    ct = contract.get("commercial_terms", {})
    row["amount"] = ct.get("amount_value")
    row["currency"] = ct.get("amount_currency")
    row["billing_frequency"] = ct.get("billing_frequency")
    row["late_fee"] = ct.get("late_fee")
    row["late_interest"] = ct.get("late_interest")

    # Timeline
    tl = contract.get("timeline", {})
    row["term_months"] = tl.get("term_months")

    # Legal
    lt = contract.get("legal_terms", {})
    row["jurisdiction"] = lt.get("jurisdiction")

    # Listenfelder → String
    row["risk_flags"] = "; ".join(contract.get("risk_flags", []))
    row["obligations_customer"] = "; ".join(contract.get("obligations_customer", []))
    row["obligations_vendor"] = "; ".join(contract.get("obligations_vendor", []))

    return row

def contracts_dict_to_df(contracts_dict):
    rows = []

    for contract_id, contract in contracts_dict.items():
        row = flatten_contract(contract, contract_id)
        rows.append(row)

    return pd.DataFrame(rows)

# Beispiel: mehrere Verträge
df = contracts_dict_to_df(contract_data).drop(columns = ['file_name'])

df.tail(15)

,contract_id,contract_type,language,signature_date,confidence,borrower_name,borrower_country,borrower_address,lender_name,lender_country,lender_address,amount,currency,billing_frequency,late_fee,late_interest,term_months,jurisdiction,risk_flags,obligations_customer,obligations_vendor
15,Contract_1,None,EN,2025-03-07,1.00,Carlos Brown,US,"1500 Brazil Avenue - Boston, MA",FINANCIAL BANK OF AMERICA Inc.,US,"1000 Main Avenue, New York, NY",82437.0,USD,monthly,2%,1% per month,60,"Courts of New York, NY",Penalties for payment delays; Automatic enforc...,Pay monthly installments increased by interest...,Grant financing to the BORROWER for the acquis...
16,Contract_12,None,EN,2024-12-30,0.95,Richard Taylor,US,"1000 Main Avenue - Chicago, IL",FINANCIAL BANK OF AMERICA Inc.,US,"1000 Main Avenue, New York, NY",83400.0,USD,monthly,2%,1% per month,60,"Courts of New York, NY",Execution of the guarantee if payment is delay...,Pay monthly installments increased by interest...,Provide financing to the BORROWER for the acqu...
17,Contract_23,None,EN,2023-09-27,1.00,Julia Miller,US,"321 Marshal Street - San Francisco, CA",FINANCIAL BANK OF AMERICA Inc.,US,"1000 Main Avenue, New York, NY",76694.0,USD,monthly,2%,1% per month,60,"Courts of New York, NY",,pay monthly installments increased by interest...,
18,Contract_22,None,EN,2024-11-06,0.95,John Smith,US,"321 Marshal Street - San Francisco, CA",FINANCIAL BANK OF AMERICA Inc.,US,"1000 Main Avenue, New York, NY",46149.0,USD,monthly,2%,1% per month,60,"Courts of New York, NY",Late payment penalties (2%) and late interest ...,Pay monthly installments; Contract mandatory i...,Provide financing to the borrower
19,Contract_20,None,EN,2024-03-16,0.95,John Smith,US,"123 Flower Street - New York, NY",FINANCIAL BANK OF AMERICA Inc.,US,"1000 Main Avenue, New York, NY",40287.0,USD,monthly,2%,1% per month,60,"The Courts of New York, NY",Late payments can result in penalties and addi...,Pay monthly installments increased by interest...,Grant financing to the BORROWER for the acquis...
20,Contract_21,None,EN,2024-12-03,1.00,Carlos Brown,US,"222 Sunshine Road - Houston, TX",FINANCIAL BANK OF AMERICA Inc.,US,"1000 Main Avenue, New York, NY",44257.0,USD,monthly,2% on the installment,1% per month,60,"Courts of New York, NY",,Pay monthly installments with interest and ann...,Grant financing to the BORROWER for the acquis...
21,Contract_19,None,EN,2024-12-31,0.95,John Smith,US,"222 Sunshine Road - Houston, TX",FINANCIAL BANK OF AMERICA Inc.,US,"1000 Main Avenue, New York, NY",84522.0,USD,monthly,2%,1% per month,60,"Courts of New York, NY",,Pay monthly installments increased by interest...,
22,Contract_25,None,EN,2025-04-16,0.90,Anna Davis,US,"123 Flower Street - New York, NY",FINANCIAL BANK OF AMERICA Inc.,US,"1000 Main Avenue, New York, NY",1195379.0,USD,monthly,2%,1% per month,60,"Courts of New York, NY",High risk of penalties for payment delay (pena...,Pay monthly installments increased by interest...,Provide financing to BORROWER as specified.
23,Contract_30,None,EN,2024-09-14,0.95,Carlos Brown,US,"789 Palm Street - Miami, FL",FINANCIAL BANK OF AMERICA Inc.,US,"1000 Main Avenue, New York, NY",83829.0,USD,monthly,2%,1% per month,60,"Courts of New York, NY",Late payment penalties: 2%; Late interest rate...,Pay monthly installments increased by interest...,Grant financing to BORROWER for acquisition of...
24,Contract_24,None,EN,2024-05-20,1.00,Emily Clark,US,"321 Marshal Street - San Francisco, CA",FINANCIAL BANK OF AMERICA Inc.,US,"1000 Main Avenue, New York, NY",1200028.0,USD,monthly,2%,1% per month,60,"Courts of New York, NY",,Pay monthly installments; Contract mandatory i...,Provide financing up to 60 months; Maintain fi...
